In [ ]:
%reload_ext autoreload
%autoreload 2

from rsg_phot.rsg_cat import get_filters, map_columns, save_photfiles, mw_extinction
import numpy as np
import glob, os
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.stats import sigma_clipped_stats as scs
from astropy import wcs
import astropy.units as u
%matplotlib inline

### Read photometry

In [ ]:
data_dir = 'data/dolphot/ngc5643_phot'
photfiles = sorted(glob.glob(os.path.join(data_dir, '*.phot')))

In [ ]:
photfiles

In [ ]:
file = photfiles[0]
colfile = file.replace('.phot', '.phot.columns')
col_idx, filters, filt_idx = map_columns(colfile)

In [ ]:
col_idx

In [ ]:
save_photfiles(data_dir, outdir=os.path.join(data_dir, 'proc'))

In [ ]:
df = pd.read_csv(os.path.join(data_dir, 'proc/ngc5643_0_0.csv'), header = None)
df

In [ ]:
refim = fits.getdata(os.path.join(data_dir, 'coadd_1_0_f150w_i2d.fits'))
me, md, std = scs(refim)
plt.imshow(refim, vmin = me - 3*std, vmax = me+3*std, origin = 'lower')
plt.scatter(df[col_idx['X']], df[col_idx['Y']], s= 0.1, color = 'red')
plt.xlim(2000, 2900)
plt.ylim(2000, 2900)
plt.show()

In [ ]:
col, mag =[], []
for clf in tqdm(glob.glob(os.path.join(data_dir, 'proc/*csv'))):
    try:
        _df = pd.read_csv(clf, header = None)
        f150_mag, f277_mag = _df[col_idx['F090W_mag']], _df[col_idx['F150W_mag']]
        col.extend(f150_mag-f277_mag)
        mag.extend(f277_mag)
    except:
        pass

In [ ]:
f090_mag, f150_mag, f277_mag =[], [], []
for clf in tqdm(glob.glob(os.path.join(data_dir, 'proc/*csv'))):
    try:
        _df = pd.read_csv(clf, header = None)
        f090_mag.extend(_df[col_idx['F090W_mag']])
        f150_mag.extend(_df[col_idx['F150W_mag']])
        f277_mag.extend(_df[col_idx['F277W_mag']])
    except:
        pass

In [ ]:
mags = np.vstack([f090_mag, f150_mag, f277_mag])

In [ ]:
np.savetxt('ngc5643_mags.txt', mags)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (12, 8), dpi = 1000)
f1, f2 = 'F090W', 'F150W'

hist = ax.hist2d(col, mag, bins = 5000, norm = 'log', cmap = 'inferno')
cbar = fig.colorbar(hist[3], ax = ax, pad = 0.009)
ax.set_xlabel(rf'$m({f1}) - m({f2})$', fontsize = 32)
ax.set_ylabel(rf'$m({f2})$', fontsize = 32)
ax.set_title('NGC5643 CMD', fontsize = 32)

# ax.legend(loc = 'lower right', fontsize = 25, framealpha = 0)
ax.tick_params(axis='both', which='major', labelsize=27)
ax.tick_params(axis='both', which='minor', labelsize=27)
cbar.ax.tick_params(labelsize=27) 
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-1, 2)
ax.set_ylim(31, 19)
plt.show()

### Index photometry

In [ ]:
data_dir = 'data/dolphot/ngc1365/phot'
photfiles = sorted(glob.glob(os.path.join(data_dir, '*.phot')))
refimgs = sorted(glob.glob(os.path.join(data_dir, '*i2d.fits')))

In [ ]:
save_photfiles(data_dir, os.path.join(data_dir, 'proc'))

In [ ]:
file = photfiles[0]
gid = '_'.join(os.path.basename(file).split('.phot')[0].split('_')[1:])
refimg = refimgs[0]
refwcs = wcs.WCS(fits.open(refimg)[1].header)
colfile = file.replace('.phot', '.phot.columns')
col_idx, filters, filt_idx = map_columns(colfile)

In [ ]:
photdf = pd.read_csv(file, sep = r'\s+', memory_map = True, 
                     header = None, iterator = True, chunksize = 100000)

In [ ]:
lc=False
for j, _df in tqdm(enumerate(photdf)):
    #apply cuts
    cuts = (_df[col_idx['SNR']] >= 5) & \
            ((_df[col_idx['Sharpness']])**2 <= 0.04) & \
            (_df[col_idx['Crowding']] <= 1.5) & \
            (_df[col_idx['Type']] <= 2)
    _df = _df[cuts].copy()

    _ra, _dec = refwcs.all_pix2world(_df[col_idx['X']], _df[col_idx['Y']], 0)
    _df.loc[:, 'RA'] = _ra
    _df.loc[:, 'Dec'] = _dec
    _df.loc[:, 'gid'] = gid
    _df.loc[:, 'id'] = np.array(_df.index)

    if lc:
        pass
        #BUG: add mw extinction correction to lc mode
    else:
        _df = _df[list(col_idx.values()) + ['RA', 'Dec', 'gid', 'id']]
        _df.columns = list(col_idx.keys()) + ['RA', 'Dec', 'gid', 'id']
        _df = mw_extinction(_df, filters, verbose=True)

    # _df.to_csv(f"{data_dir}/{os.path.basename(file).split('.')[0]}_{j}.csv", 
    #             mode = 'a', header = True, index = False)

In [ ]:
cat = None
for fl in sorted(glob.glob(os.path.join(data_dir, '*csv'))):
    if cat is None:
        cat = pd.read_csv(fl)
    else:
        df_ = pd.read_csv(fl)
        cat = pd.concat([cat, df_])
cat.reset_index(inplace=True)